# Unit 14 solutions: the full system (what earns its place)

These reference solutions **use** every heavy piece of Part 2 inside the assembled recommender and measure what each one buys: the HNSW index, the feature tower on cold books, hard negatives, the learned reranker, and self-attention on the session log. Then they read the ablation table, the beyond-accuracy table, the Part-1 benchmark and the hit-vs-reach frontier.

The honest headline is the lesson's: **no system dominates**. The full stack (about 0.356 hit@10) sits just below the fixed 8-path blend (about 0.362), the content-only reranker has the best hit but the narrowest reach, and the oldest build, the Part-1 blend, still reaches furthest into the catalog. Choosing among them is a product decision, not a metric's verdict.

Every number is measured on the **validation** split over the scoreboard's 500-reader val cohort (the sequence path on the session log's own 1,419-reader val cohort). The sealed held-out split is never touched. **Nothing expensive is fitted**: every MF, tower, index source and SASRec model loads from the cache, and the only training here is the reranker's tiny MLP refit on its cached training matrix.

In [1]:
import time

import numpy as np
import pandas as pd
from bookrec import (
    AnnRetrievalPath,
    FixedBlendPath,
    FullSystem,
    HnswIndex,
    ItemItemRetrievalPath,
    NeuralRerankerPath,
    RerankerModel,
    SasrecRetrievalPath,
    catalog_coverage,
    load_book_data,
    load_cached_models,
)
from bookrec.rerank import family_mask, feature_names
from bookrec.system import FULL8_PATH_NAMES, top_k

SEED = 0         # every cached model was trained with seed 0; the reranker refits below use it too
K = 10           # every readout is at the top-10 cutoff
EF_SEARCH = 64   # the served HNSW index's pinned efSearch (Unit 10)

# --- the generated data and the model cache (loads only; no expensive refit) ----------------
data = load_book_data()                              # catalog, keywords, GloVe, main log, val cohort
cache = load_cached_models(data_dir=data.data_dir)   # every expensive model's artifact + readouts.json
system = FullSystem.from_cache(cache, data)          # cheap paths refit in seconds; the rest loaded
paths = system.paths                                 # {name: fitted path}, e.g. "two-tower", "ann", "feature-tower"
cohort = data.cohort                                 # the 500-reader main-log val cohort (ascending ids)

# Books with NO train positive (818 of the 2,000) -- the cold-item slice of Exercises 2, 3 and 9.
zero_train = set(data.catalog_ids) - set(paths["popularity"].counts)

# system.top_lists(path)  -> {reader: top-10 item ids} over the cohort (seen books excluded)
# system.readout(lists)   -> {"hit", "coverage", "head_share", "exposure_gini", "author_gini",
#                             "novelty", "diversity"} of such a mapping
# system.evaluate(path)   -> system.readout(system.top_lists(path))
# system.full_stack       -> the 8-path pool + learned reranker (two-tower served through HNSW)
# system.fixed_blend(drop=None) -> the equal-weight 8-path fixed blend (drop one path for leave-one-out)
# system.full8            -> the 8 serving paths, {name: path}, in FULL8_PATH_NAMES order


def cold_in(lists):
    # The DISTINCT zero-train books that appear anywhere in a {reader: top-10} mapping.
    return {item for top in lists.values() for item in top} & zero_train

## Exercise 1 - The served HNSW index: an `efSearch` and `M` sweep (recall vs time)

I time one batched search over all 500 cohort readers at each `efSearch` and score recall@10 against the exact inner-product scan. Recall climbs from about 0.73 at `efSearch=1` to essentially 1.0 by 64, and the search time grows with the breadth. Fresh indexes with a smaller `M` are less faithful at the same `efSearch=16`. Finally, a separate copy of the ANN path shows that the system's hit@10 barely moves with recall: the pinned value of 64 reproduces the exact two-tower's 0.340.

In [2]:
# Given: the exact brute-force reference, from the cached ANN artifact (the two-tower's vectors).
ann_state = cache.artifact("ann")
item_matrix = np.array(ann_state["index"]["items"], dtype=np.float32)   # (2000, dim) item vectors
item_ids = list(ann_state["index"]["ids"])                              # aligned to the index rows
row_of = {reader: i for i, reader in enumerate(ann_state["reader_ids"])}
queries = np.array(ann_state["reader_embeddings"], dtype=np.float32)[[row_of[r] for r in cohort]]
exact_rows = np.argsort(-(queries @ item_matrix.T), axis=1)[:, :K]       # exact top-10 ROW indices


def recall_and_time(index, ef):
    # (mean recall@10 vs exact over the cohort, seconds for ONE batched search at efSearch=ef).
    start = time.perf_counter()
    labels, _ = index.search(queries, K, efSearch=ef)
    seconds = time.perf_counter() - start
    overlaps = [
        len(set(labels[i].tolist()) & set(exact_rows[i].tolist())) / K for i in range(len(queries))
    ]
    return float(np.mean(overlaps)), seconds


served_index = paths["ann"].index   # the full stack's HNSW index (M = 32, efSearch pinned at 64)
EF_SWEEP = [1, 4, 16, 64, 256]
ef_rows = []
for ef in EF_SWEEP:
    recall, seconds = recall_and_time(served_index, ef)
    ef_rows.append({"efSearch": ef, "recall": recall, "seconds": seconds})
ef_table = pd.DataFrame(ef_rows)
recall_of = dict(zip(ef_table["efSearch"], ef_table["recall"]))
seconds_of = dict(zip(ef_table["efSearch"], ef_table["seconds"]))
recall_rises = bool(recall_of[1] < recall_of[256])
pinned_recall = float(recall_of[EF_SEARCH])
pinned_is_faithful = bool(pinned_recall >= 0.995)
slower_when_broader = bool(seconds_of[256] > seconds_of[1])

M_SWEEP = [4, 8, 32]
m_table = pd.DataFrame([
    {"M": m, "recall_ef16": recall_and_time(HnswIndex(item_matrix, item_ids, m=m), 16)[0]}
    for m in M_SWEEP
])
m_recall_of = dict(zip(m_table["M"], m_table["recall_ef16"]))
m_recall_rises = bool(m_recall_of[4] < m_recall_of[32])

sweep_ann = AnnRetrievalPath(paths["two-tower"]).load(ann_state)   # a SEPARATE copy for the sweep
ann_hit = {}
for ef in (4, 64):
    sweep_ann.efSearch = ef
    ann_hit[ef] = system.evaluate(sweep_ann)["hit"]
exact_hit = system.evaluate(paths["two-tower"])["hit"]
pinned_matches_exact = bool(abs(ann_hit[64] - exact_hit) <= 0.002)
hit_moves_little = bool(abs(ann_hit[4] - ann_hit[64]) <= 0.01)

ann_explanation = (
    "efSearch trades query time for recall at query time (one batched search over 500 readers grows "
    "from a few milliseconds to tens), while M trades build time and memory for a better-connected "
    "graph that is more faithful at the same efSearch (0.85 at M=4 vs 0.996 at M=32). At the served "
    "efSearch of 64 the index returns essentially the exact top-10 (recall about 0.999) for a few "
    "milliseconds, and the system's hit equals the exact scan's 0.340, so 64 buys fidelity cheaply. "
    "Recall measures FIDELITY to the exact scan: a lower-recall index only reorders near-ties, and a "
    "0.006 higher hit at efSearch=4 is 3 readers of 500, sampling noise rather than a reason to serve "
    "a less faithful index."
)

print(ef_table.round(4))
print(m_table.round(3))
print(f"pinned recall at efSearch={EF_SEARCH}: {pinned_recall:.4f} | recall rises (ef 1 -> 256): {recall_rises} | "
      f"slower when broader: {slower_when_broader} | M recall rises (4 -> 32): {m_recall_rises}")
print(f"system hit@10: ANN efSearch=4 {ann_hit[4]:.3f}, efSearch=64 {ann_hit[64]:.3f}, exact scan {exact_hit:.3f}")
print(ann_explanation)

assert recall_rises and m_recall_rises, "recall rises with efSearch and with M (endpoints)"
assert pinned_is_faithful and abs(recall_of[1] - 0.733) < 0.01 and abs(m_recall_of[4] - 0.845) < 0.01
assert pinned_matches_exact and abs(exact_hit - 0.340) < 0.0011, "the served index reproduces the exact two-tower"
assert hit_moves_little and abs(ann_hit[4] - 0.346) < 0.0011
assert paths["ann"].efSearch == EF_SEARCH, "the served ANN path was never changed"

   efSearch  recall  seconds
0         1  0.7334   0.0022
1         4  0.9326   0.0030
2        16  0.9960   0.0059
3        64  0.9990   0.0160
4       256  1.0000   0.0520
    M  recall_ef16
0   4        0.845
1   8        0.953
2  32        0.996
pinned recall at efSearch=64: 0.9990 | recall rises (ef 1 -> 256): True | slower when broader: True | M recall rises (4 -> 32): True
system hit@10: ANN efSearch=4 0.346, efSearch=64 0.340, exact scan 0.340
efSearch trades query time for recall at query time (one batched search over 500 readers grows from a few milliseconds to tens), while M trades build time and memory for a better-connected graph that is more faithful at the same efSearch (0.85 at M=4 vs 0.996 at M=32). At the served efSearch of 64 the index returns essentially the exact top-10 (recall about 0.999) for a few milliseconds, and the system's hit equals the exact scan's 0.340, so 64 buys fidelity cheaply. Recall measures FIDELITY to the exact scan: a lower-recall index only re

## Exercise 2 - A cold-item slice: does the feature tower's cold reach survive the stack?

Unit 9's readout reproduces from the cache: over every fitted, non-cold reader the feature tower surfaces 113 of the 818 zero-train books and the ID-only two-tower none. Then I follow the cold books through the system on the cohort. The feature tower's own lists hold 109 of them; the fixed blend keeps only 23 (in 49 of its 5,000 slots), and just 15 of those came from the feature tower's set; dropping the feature tower from the blend gives 27; the reranked full stack shows **none**.

In [3]:
cold_readers = set(data.cold_readers)


def cold_coverage(path):
    # The number of distinct zero-train books in the top-10 of ANY fitted, non-cold reader of `path`.
    surfaced = set()
    readers = [int(r) for r in path.artifact()["reader_ids"] if int(r) not in cold_readers]
    for reader in readers:
        for c in path.retrieve(reader, {"seen": data.seen_by.get(reader, set())}, K):
            if c.item_id in zero_train:
                surfaced.add(c.item_id)
    return len(surfaced)


feature_tower_cold = cold_coverage(paths["feature-tower"])
two_tower_cold = cold_coverage(paths["two-tower"])

feature_tower_lists = system.top_lists(paths["feature-tower"])
blend_lists = system.top_lists(system.fixed_blend())   # the first full-pool pass (~20 s)
blend_without_ft_lists = system.top_lists(system.fixed_blend(drop="feature-tower"))
stack_lists = system.top_lists(system.full_stack)

cold_slice = {
    "feature tower": len(cold_in(feature_tower_lists)),
    "fixed blend": len(cold_in(blend_lists)),
    "fixed blend without feature tower": len(cold_in(blend_without_ft_lists)),
    "full stack": len(cold_in(stack_lists)),
}
blend_cold_slots = sum(1 for top in blend_lists.values() for item in top if item in zero_train)
blend_cold_from_ft = len(cold_in(blend_lists) & cold_in(feature_tower_lists))

feature_tower_reaches_cold = bool(feature_tower_cold > 0 and two_tower_cold == 0)
stack_shows_no_cold = bool(cold_slice["full stack"] == 0)

cold_explanation = (
    "Only the content paths (lexical, semantic, and the hybrid's lexical half) and the feature tower "
    "can score a zero-train book at all; in the calibrated union a cold book usually carries one vote "
    "against warm books with several, so only 23 reach a blended top-10. Most of those come from the "
    "lexical and semantic paths (just 15 overlap the feature tower's 109), so dropping the feature "
    "tower does not lower the count -- it frees a few slots for the content paths' cold books (27). "
    "The reranker learned from logged reads that popular books win (the log_pop feature, Exercise 9), "
    "so it ranks every cold book out; a system that must serve new books needs a reserved cold slot "
    "or a cold-aware ranker, not just a cold-capable path in the pool."
)

print(f"cold coverage over every fitted reader: feature tower {feature_tower_cold} of {len(zero_train)}, "
      f"two-tower {two_tower_cold}")
for name, count in cold_slice.items():
    print(f"  {name:<34} {count:>4} distinct zero-train books in the cohort's top-10s")
print(f"fixed blend: {blend_cold_slots} cold slots of {K * len(cohort)} | "
      f"{blend_cold_from_ft} of its cold books are among the feature tower's")
print(cold_explanation)

assert feature_tower_reaches_cold and (feature_tower_cold, len(zero_train)) == (113, 818)
assert cold_slice == {"feature tower": 109, "fixed blend": 23, "fixed blend without feature tower": 27,
                      "full stack": 0}, cold_slice
assert stack_shows_no_cold and (blend_cold_slots, blend_cold_from_ft) == (49, 15)
assert cold_slice["fixed blend without feature tower"] >= cold_slice["fixed blend"], \
    "dropping the feature tower does not lower the blend's cold count"

cold coverage over every fitted reader: feature tower 113 of 818, two-tower 0
  feature tower                       109 distinct zero-train books in the cohort's top-10s
  fixed blend                          23 distinct zero-train books in the cohort's top-10s
  fixed blend without feature tower    27 distinct zero-train books in the cohort's top-10s
  full stack                            0 distinct zero-train books in the cohort's top-10s
fixed blend: 49 cold slots of 5000 | 15 of its cold books are among the feature tower's
Only the content paths (lexical, semantic, and the hybrid's lexical half) and the feature tower can score a zero-train book at all; in the calibrated union a cold book usually carries one vote against warm books with several, so only 23 reach a blended top-10. Most of those come from the lexical and semantic paths (just 15 overlap the feature tower's 109), so dropping the feature tower does not lower the count -- it frees a few slots for the content paths' cold 

## Exercise 3 - Hard negatives: the cached negative-pool ablation

Both towers load from the cache, so the ablation is a pure readout. The hard-negative tower gives up about 0.126 hit@10 and in exchange widens coverage from 0.307 to 0.450, more than halves the head-share, lowers the exposure Gini, raises novelty, and surfaces 185 zero-train books instead of 113. The two final losses (about 0.069 and 0.071) are recorded but are not a verdict: they are losses on different negative sets.

In [4]:
hard_tower = paths["feature-tower-hard"]   # loaded from the cache -- do NOT refit a feature tower

warm_readout = system.evaluate(paths["feature-tower"])
hard_readout = system.evaluate(hard_tower)
COLUMNS = ("hit", "coverage", "head_share", "exposure_gini", "novelty")
negative_pool_table = pd.DataFrame([
    {"pool": "warm", **{c: warm_readout[c] for c in COLUMNS}, "cold_coverage": feature_tower_cold},
    {"pool": "hard", **{c: hard_readout[c] for c in COLUMNS}, "cold_coverage": cold_coverage(hard_tower)},
])
pool_rows = {row["pool"]: row for row in negative_pool_table.to_dict("records")}

final_losses = {name: cache.loss_history(name)[-1] for name in ("feature-tower", "feature-tower-hard")}

hard_costs_warm_hit = bool(hard_readout["hit"] < warm_readout["hit"])
hard_widens_reach = bool(
    hard_readout["coverage"] > warm_readout["coverage"]
    and pool_rows["hard"]["cold_coverage"] > pool_rows["warm"]["cold_coverage"]
)
hard_leaves_the_head = bool(hard_readout["head_share"] < warm_readout["head_share"])

hard_negative_explanation = (
    "Hard negatives cost about 0.126 hit@10 (0.298 -> 0.172) and buy reach: coverage 0.307 -> 0.450, "
    "head-share 0.638 -> 0.275, and 185 cold books instead of 113. Sampling popular books as negatives "
    "teaches the tower to score the head DOWN for every reader, so its lists spread toward the tail -- "
    "and a popular unread book may be a false negative the reader would have liked. The two final "
    "losses are averages over different negative sets, so they measure different tasks and cannot "
    "rank the towers; only a shared readout can. The stack serves the warm tower because a 0.126 hit "
    "cost is too large for one of eight paths, and the content paths already supply reach."
)

print(negative_pool_table.round(3))
print("final training losses: " + ", ".join(f"{n} {v:.3f}" for n, v in final_losses.items()))
print(f"hard costs hit: {hard_costs_warm_hit} | widens reach: {hard_widens_reach} | "
      f"leaves the head: {hard_leaves_the_head}")
print(hard_negative_explanation)

assert hard_costs_warm_hit and hard_widens_reach and hard_leaves_the_head
assert abs(warm_readout["hit"] - 0.298) < 0.0011 and abs(hard_readout["hit"] - 0.172) < 0.0011
assert pool_rows["hard"]["cold_coverage"] == 185 and pool_rows["warm"]["cold_coverage"] == 113
assert abs(final_losses["feature-tower"] - 0.069) < 0.002 and abs(final_losses["feature-tower-hard"] - 0.071) < 0.002

   pool    hit  coverage  head_share  exposure_gini  novelty  cold_coverage
0  warm  0.298     0.306       0.638          0.877    8.936            113
1  hard  0.172     0.450       0.275          0.777   10.252            185
final training losses: feature-tower 0.069, feature-tower-hard 0.071
hard costs hit: True | widens reach: True | leaves the head: True
Hard negatives cost about 0.126 hit@10 (0.298 -> 0.172) and buy reach: coverage 0.307 -> 0.450, head-share 0.638 -> 0.275, and 185 cold books instead of 113. Sampling popular books as negatives teaches the tower to score the head DOWN for every reader, so its lists spread toward the tail -- and a popular unread book may be a false negative the reader would have liked. The two final losses are averages over different negative sets, so they measure different tasks and cannot rank the towers; only a shared readout can. The stack serves the warm tower because a 0.126 hit cost is too large for one of eight paths, and the content paths

## Exercise 4 - Learning to rank: a feature-family mask and refit on the cached training matrix

The cached training matrix is 7,458 x 21 with a positive rate of about 0.091, built with the Unit-11 holdout recipe, and refitting the MLP on all features reproduces the cached weights exactly. Masking whole feature families shows where the hit lives: the **content** family alone gives the best hit (0.384) and the narrowest, most head-heavy lists; dropping content barely changes the hit of the full model (0.354 vs 0.356).

In [5]:
def with_model(model):
    # Given: a fresh full-stack reranker over the same 8 serving paths, with `model` as its ordering key.
    # (It loads the cached serving statistics; its refit factory is never called.)
    reranker = NeuralRerankerPath(
        system.full8, lambda rows: {}, data.catalog, pool=50, seed=SEED, path_names=FULL8_PATH_NAMES
    ).load(cache.artifact("reranker-full8"))
    return reranker.set_model(model)


X, y = cache.training_matrix("reranker-full8")
matrix_shape = X.shape
positive_rate = float(y.mean())
recipe = cache.meta("reranker-full8")["recipe"]
names = feature_names(FULL8_PATH_NAMES)

refit_model = RerankerModel(seed=SEED).fit(X, y)
cached_weights = cache.artifact("reranker-full8")["model"]["weights"]
refit_matches_cache = bool(
    len(refit_model.weights) == len(cached_weights)
    and all(np.allclose(a, b) for a, b in zip(refit_model.weights, cached_weights))
)

FAMILY_VARIANTS = {
    "all": ("scores", "presence", "meta", "content"),
    "content only": ("content",),
    "no content": ("scores", "presence", "meta"),
}
variant_models = {
    name: RerankerModel(seed=SEED).fit(X, y, feature_mask=family_mask(families, FULL8_PATH_NAMES))
    for name, families in FAMILY_VARIANTS.items()
}
family_rows = []
for name, model in variant_models.items():
    readout = system.evaluate(with_model(model))
    family_rows.append({"variant": name, **{c: readout[c] for c in ("hit", "coverage", "head_share",
                                                                    "exposure_gini")}})
family_table = pd.DataFrame(family_rows)
variant_of = {row["variant"]: row for row in family_table.to_dict("records")}

content_has_best_hit = bool(variant_of["content only"]["hit"] == family_table["hit"].max())
content_is_narrowest = bool(
    variant_of["content only"]["coverage"] == family_table["coverage"].min()
    and variant_of["content only"]["head_share"] == family_table["head_share"].max()
)
scores_dilute_content = bool(variant_of["all"]["hit"] < variant_of["content only"]["hit"])

ranking_explanation = (
    "The content family (genre and author affinity plus log-popularity) carries the hit: alone it "
    "scores 0.384, while the three other families together score 0.354, almost the full model's "
    "0.356. Content sees a cross-reader taste-and-popularity signal that the per-list calibrated path "
    "scores cannot, and it buys that hit with reach (coverage 0.137, head-share 0.96). The 16 path "
    "scores and flags are correlated, noisy features that the small MLP also leans on, so mixing them "
    "in pulls the ranking back toward the blend -- the all-feature model is a dilution of the "
    "content-only one, not an improvement on it."
)

print(f"training matrix {matrix_shape}, positive rate {positive_rate:.3f}, recipe {recipe!r}")
print(f"{len(names)} features: {names}")
print(f"refit on all features matches the cached weights: {refit_matches_cache}")
print(family_table.round(4))
print(f"content best hit: {content_has_best_hit} | content narrowest: {content_is_narrowest} | "
      f"scores dilute content: {scores_dilute_content}")
print(ranking_explanation)

assert matrix_shape == (7458, 21) and abs(positive_rate - 0.091) < 0.001 and recipe == "u11-holdout-75-25"
assert refit_matches_cache, "the refit on the cached matrix reproduces the cached reranker"
assert names[-4:] == ["genre_frac", "genre_cos", "author_frac", "log_pop"], "content family comes last"
assert content_has_best_hit and content_is_narrowest and scores_dilute_content
assert abs(variant_of["all"]["hit"] - 0.356) < 0.0011 and abs(variant_of["content only"]["hit"] - 0.384) < 0.0011

training matrix (7458, 21), positive rate 0.091, recipe 'u11-holdout-75-25'
21 features: ['score:popularity', 'score:lexical', 'score:item-item', 'score:mf', 'score:semantic', 'score:two-tower', 'score:feature-tower', 'score:hybrid', 'in:popularity', 'in:lexical', 'in:item-item', 'in:mf', 'in:semantic', 'in:two-tower', 'in:feature-tower', 'in:hybrid', 'n_paths', 'genre_frac', 'genre_cos', 'author_frac', 'log_pop']
refit on all features matches the cached weights: True
        variant    hit  coverage  head_share  exposure_gini
0           all  0.356    0.2025      0.8466         0.9131
1  content only  0.384    0.1365      0.9604         0.9404
2    no content  0.354    0.1860      0.8450         0.9204
content best hit: True | content narrowest: True | scores dilute content: True
The content family (genre and author affinity plus log-popularity) carries the hit: alone it scores 0.384, while the three other families together score 0.354, almost the full model's 0.356. Content sees a cr

## Exercise 5 - Self-attention on the session log: a SASRec readout and an order probe

SASRec scores about 0.423 hit@10 on the session log's 1,419-reader cohort with coverage 0.285, and its cached curve falls from about 7.30 to 5.06 over 30 epochs. Counting item-item CF on the same session rows beats it by about 0.057 (about 5 paired SEs). The three no-refit probes show that the model reads **order** (reversing costs about 0.049) and attends mainly to a short **recent window** (the last 5 reads lose nothing; the last read alone loses about 0.062).

In [6]:
session_seen, session_relevant, session_cohort = data.session_cohort


def session_hits(lists):
    # 1.0 / 0.0 per session-cohort reader: did their top-10 hit an unseen session-log val positive?
    return np.array([1.0 if set(lists[r]) & session_relevant[r] else 0.0 for r in session_cohort])


def paired(a, b):
    # (mean of a - b, paired standard error) over the same readers.
    diffs = a - b
    return float(diffs.mean()), float(np.std(diffs, ddof=1) / np.sqrt(len(diffs)))


sasrec_state = cache.artifact("sasrec")


def probe(edit):
    # Given: SASRec's top-10 lists after editing every stored history with `edit` -- no refit.
    state = dict(sasrec_state)
    state["histories"] = {r: edit(list(h)) for r, h in sasrec_state["histories"].items()}
    edited = SasrecRetrievalPath().load(state)
    return {r: top_k(edited, r, session_seen.get(r, set())) for r in session_cohort}


session_cohort_size = len(session_cohort)
sasrec_lists = system.session_top_lists()
sasrec_hits = session_hits(sasrec_lists)
sasrec_coverage = catalog_coverage(list(sasrec_lists.values()), data.catalog_ids)
sasrec_loss = cache.loss_history("sasrec")

session_item_item = ItemItemRetrievalPath().fit(data.session_rows, catalog=data.catalog_ids)
item_item_hits = session_hits({r: top_k(session_item_item, r, session_seen.get(r, set()))
                               for r in session_cohort})
sasrec_vs_item_item = paired(sasrec_hits, item_item_hits)

EDITS = {"reversed": lambda h: h[::-1], "last 1": lambda h: h[-1:], "last 5": lambda h: h[-5:]}
probe_rows = []
for name, edit in EDITS.items():
    edited_hits = session_hits(probe(edit))
    drop, se = paired(sasrec_hits, edited_hits)
    probe_rows.append({"edit": name, "hit": float(edited_hits.mean()), "drop": drop, "se": se})
probe_table = pd.DataFrame(probe_rows)
probe_of = {row["edit"]: row for row in probe_table.to_dict("records")}

order_matters = bool(probe_of["reversed"]["drop"] > 2 * probe_of["reversed"]["se"])
recent_window_suffices = bool(abs(probe_of["last 5"]["drop"]) < 2 * probe_of["last 5"]["se"])

attention_explanation = (
    "Reversing each history keeps the same books in a new order and costs about 0.049 hit@10 at about "
    "5 SE, so the attention reads ORDER, not just the set -- a fair probe, because the cached model "
    "and the books are unchanged and only the positions move. Keeping only the last read loses more "
    "(about 0.062), but keeping the last 5 loses nothing, so the model attends mostly to a short "
    "recent window. It still trails counting item-item CF on this log by about 0.057. The session log "
    "is a different reader population with a different next-read target, so its hit@10 is not on the "
    "main log's scale and must never be compared with the 500-reader cohort's numbers."
)

print(f"session cohort: {session_cohort_size} readers | SASRec hit@10 {sasrec_hits.mean():.3f}, "
      f"coverage {sasrec_coverage:.3f}")
print(f"SASRec loss: {len(sasrec_loss)} epochs, {sasrec_loss[0]:.2f} -> {sasrec_loss[-1]:.2f}")
print(f"session item-item hit@10 {item_item_hits.mean():.3f} | SASRec - item-item "
      f"{sasrec_vs_item_item[0]:+.3f} (SE {sasrec_vs_item_item[1]:.3f})")
print(probe_table.round(3))
print(f"order matters: {order_matters} | the last 5 suffice: {recent_window_suffices}")
print(attention_explanation)

assert session_cohort_size == 1419 and len(sasrec_loss) == 30 and sasrec_loss[-1] < sasrec_loss[0]
assert abs(sasrec_hits.mean() - 0.423) < 0.0011 and abs(item_item_hits.mean() - 0.480) < 0.0011
assert sasrec_vs_item_item[0] < -2 * sasrec_vs_item_item[1], "SASRec trails counting CF on the session log"
assert order_matters and recent_window_suffices
assert probe_of["last 1"]["drop"] > 2 * probe_of["last 1"]["se"], "the last read alone is not enough"

session cohort: 1419 readers | SASRec hit@10 0.423, coverage 0.284
SASRec loss: 30 epochs, 7.30 -> 5.06
session item-item hit@10 0.480 | SASRec - item-item -0.057 (SE 0.012)
       edit    hit   drop     se
0  reversed  0.374  0.049  0.010
1    last 1  0.361  0.062  0.013
2    last 5  0.429 -0.006  0.009
order matters: True | the last 5 suffice: True
Reversing each history keeps the same books in a new order and costs about 0.049 hit@10 at about 5 SE, so the attention reads ORDER, not just the set -- a fair probe, because the cached model and the books are unchanged and only the positions move. Keeping only the last read loses more (about 0.062), but keeping the last 5 loses nothing, so the model attends mostly to a short recent window. It still trails counting item-item CF on this log by about 0.057. The session log is a different reader population with a different next-read target, so its hit@10 is not on the main log's scale and must never be compared with the 500-reader cohort's nu

## Exercise 6 - The ablation table: what each stage and each path buys

I put the 10 cached rows in a table and recompute every delta from the declared baselines; all match to 1e-9. Two rows recomputed live (the fixed blend and minus-popularity) equal the cached ones exactly. The reranker stage does not earn its place on hit (removing it gains 0.006), ANN costs one reader, the two-tower is the costliest path on hit and lexical the costliest on reach, and popularity and the feature tower can be removed for free -- but every delta is within about 5 readers of 500.

In [7]:
readouts = cache.readouts   # the build's readouts.json (val only)

ablation = pd.DataFrame([
    {key: row[key] for key in ("row", "kind", "baseline", "hit", "coverage", "d_hit", "d_coverage")}
    for row in readouts["ablation"]
])
baselines = readouts["baselines"]
deltas_recomputed = bool(all(
    abs(row["d_hit"] - (row["hit"] - baselines[row["baseline"]]["hit"])) <= 1e-9
    and abs(row["d_coverage"] - (row["coverage"] - baselines[row["baseline"]]["coverage"])) <= 1e-9
    for row in ablation.to_dict("records")
))

live_blend = system.evaluate(system.fixed_blend())
live_minus_popularity = system.evaluate(system.fixed_blend(drop="popularity"))
row_of_ablation = {row["row"]: row for row in ablation.to_dict("records")}
live_matches_cache = bool(
    abs(live_blend["hit"] - baselines["fixed-blend"]["hit"]) <= 1e-9
    and abs(live_blend["coverage"] - baselines["fixed-blend"]["coverage"]) <= 1e-9
    and abs(live_minus_popularity["hit"] - row_of_ablation["minus-popularity"]["hit"]) <= 1e-9
    and abs(live_minus_popularity["coverage"] - row_of_ablation["minus-popularity"]["coverage"]) <= 1e-9
)

stage_rows = ablation[ablation["kind"] == "stage"]
path_rows = ablation[ablation["kind"] == "path"]
path_records = path_rows.to_dict("records")
costliest_hit = min(path_records, key=lambda row: row["d_hit"])["row"]
costliest_coverage = min(path_records, key=lambda row: row["d_coverage"])["row"]
free_removals = sorted(path_rows[(path_rows["d_hit"] >= 0) & (path_rows["d_coverage"] >= 0)]["row"])

ablation_explanation = (
    "Removing the reranker RAISES hit by 0.006 and coverage by 0.0035, so the learned stage does not "
    "earn its place on this data; swapping ANN for the exact scan moves hit by -0.002 (one reader) and "
    "reach by 0, so ANN costs nothing. Among the paths the dense two-tower carries the most hit "
    "(-0.010 without it) and the lexical path the most reach (-0.031 coverage), while popularity and "
    "the feature tower can go for free. But every delta is at most 0.01 hit -- about 5 of 500 readers, "
    "well inside a hit's sampling noise -- so the table ranks pieces only roughly, and the path rows "
    "are fixed-blend leave-one-out, not what the reranked stack would do without that path."
)

print(ablation.round(4))
print(f"deltas recomputed from the baselines: {deltas_recomputed}")
print(f"live fixed blend {live_blend['hit']:.3f} / {live_blend['coverage']:.4f} | live minus-popularity "
      f"{live_minus_popularity['hit']:.3f} / {live_minus_popularity['coverage']:.4f} | match cache: {live_matches_cache}")
print(f"costliest hit: {costliest_hit} | costliest coverage: {costliest_coverage} | free: {free_removals}")
print(ablation_explanation)

stage_of = {row["row"]: row for row in stage_rows.to_dict("records")}
assert len(ablation) == 10 and len(stage_rows) == 2 and len(path_rows) == 8
assert deltas_recomputed and live_matches_cache
assert abs(stage_of["no-reranker"]["d_hit"] - 0.006) < 1e-9 and abs(stage_of["exact-two-tower"]["d_hit"] + 0.002) < 1e-9
assert (costliest_hit, costliest_coverage) == ("minus-two-tower", "minus-lexical")
assert free_removals == ["minus-feature-tower", "minus-popularity"]

                   row   kind     baseline    hit  coverage  d_hit  d_coverage
0          no-reranker  stage   full-stack  0.362    0.2060  0.006      0.0035
1      exact-two-tower  stage   full-stack  0.354    0.2025 -0.002      0.0000
2     minus-popularity   path  fixed-blend  0.370    0.2200  0.008      0.0140
3        minus-lexical   path  fixed-blend  0.362    0.1750  0.000     -0.0310
4      minus-item-item   path  fixed-blend  0.354    0.1920 -0.008     -0.0140
5             minus-mf   path  fixed-blend  0.354    0.2375 -0.008      0.0315
6       minus-semantic   path  fixed-blend  0.358    0.1855 -0.004     -0.0205
7      minus-two-tower   path  fixed-blend  0.352    0.2210 -0.010      0.0150
8  minus-feature-tower   path  fixed-blend  0.364    0.2075  0.002      0.0015
9         minus-hybrid   path  fixed-blend  0.356    0.2105 -0.006      0.0045
deltas recomputed from the baselines: True
live fixed blend 0.362 / 0.2060 | live minus-popularity 0.370 / 0.2200 | match cache: Tr

## Exercise 7 - Beyond accuracy, the Part-1 benchmark, and the frontier

The beyond-accuracy table has the three expected rows, and the Part-1 benchmark recomputed live matches its cached row exactly. On the hit-vs-coverage plane the frontier is **Part-1 build, fixed blend, content-only reranker**: the fixed blend dominates both the full stack and the hybrid (best single path), so the full system's extra machinery sits inside the frontier.

In [8]:
beyond = pd.DataFrame(readouts["beyond_accuracy"])[["system", "hit", "coverage", "head_share",
                                                     "exposure_gini", "novelty", "diversity"]]
beyond_of = {row["system"]: row for row in beyond.to_dict("records")}
part1_live = system.evaluate(system.part1_build())
part1_matches = bool(
    abs(part1_live["hit"] - beyond_of["part1-build"]["hit"]) <= 1e-9
    and abs(part1_live["coverage"] - beyond_of["part1-build"]["coverage"]) <= 1e-9
)

points = {
    "part1-build": (beyond_of["part1-build"]["hit"], beyond_of["part1-build"]["coverage"]),
    "best-single-path": (beyond_of["best-single-path"]["hit"], beyond_of["best-single-path"]["coverage"]),
    "fixed-blend": (baselines["fixed-blend"]["hit"], baselines["fixed-blend"]["coverage"]),
    "full-stack": (baselines["full-stack"]["hit"], baselines["full-stack"]["coverage"]),
    "content-reranker": (variant_of["content only"]["hit"], variant_of["content only"]["coverage"]),
}


def dominated(name, points):
    # True if some OTHER system is >= on both hit and coverage and > on at least one.
    hit, cov = points[name]
    return any(h >= hit and c >= cov and (h > hit or c > cov)
               for other, (h, c) in points.items() if other != name)


frontier = [name for name in points if not dominated(name, points)]
full_stack_dominated = bool(dominated("full-stack", points))
part1_widest_reach = bool(
    beyond_of["part1-build"]["coverage"] == beyond["coverage"].max()
    and beyond_of["part1-build"]["exposure_gini"] == beyond["exposure_gini"].min()
)

frontier_explanation = (
    "The frontier is a menu of three: the Part-1 build reaches furthest (coverage 0.333, the lowest "
    "Gini, the most diverse lists) at the lowest hit (0.306); the fixed blend is the balanced point "
    "(0.362 / 0.206); the content-only reranker has the best hit (0.384) and is the most concentrated "
    "(coverage 0.137, head-share 0.96). The full stack adds a learned stage but sits inside the "
    "frontier, dominated by the simpler fixed blend. Head-share, Gini, novelty and diversity say WHO "
    "gets the slots and how varied a single list is, which a count of distinct books hides. Choosing a "
    "point is a product and values decision -- for the people who answer to readers and authors -- "
    "not a metric's verdict."
)

print(beyond.round(4))
print(f"Part-1 live {part1_live['hit']:.3f} / {part1_live['coverage']:.4f} | matches cache: {part1_matches}")
for name, (h, c) in points.items():
    print(f"  {name:<17} hit {h:.3f}  coverage {c:.4f}  dominated: {dominated(name, points)}")
print(f"frontier: {frontier} | Part-1 widest reach: {part1_widest_reach}")
print(frontier_explanation)

assert list(beyond["system"]) == ["full-system", "best-single-path", "part1-build"] and part1_matches
assert frontier == ["part1-build", "fixed-blend", "content-reranker"], frontier
assert full_stack_dominated and dominated("best-single-path", points)
assert part1_widest_reach and abs(part1_live["hit"] - 0.306) < 0.0011

             system    hit  coverage  head_share  exposure_gini  novelty  \
0       full-system  0.356    0.2025      0.8466         0.9131   8.5677   
1  best-single-path  0.362    0.1920      0.8616         0.9220   8.4825   
2       part1-build  0.306    0.3330      0.7930         0.9080   8.4915   

   diversity  
0     0.8566  
1     0.8501  
2     0.8711  
Part-1 live 0.306 / 0.3330 | matches cache: True
  part1-build       hit 0.306  coverage 0.3330  dominated: False
  best-single-path  hit 0.362  coverage 0.1920  dominated: True
  fixed-blend       hit 0.362  coverage 0.2060  dominated: False
  full-stack        hit 0.356  coverage 0.2025  dominated: True
  content-reranker  hit 0.384  coverage 0.1365  dominated: False
frontier: ['part1-build', 'fixed-blend', 'content-reranker'] | Part-1 widest reach: True
The frontier is a menu of three: the Part-1 build reaches furthest (coverage 0.333, the lowest Gini, the most diverse lists) at the lowest hit (0.306); the fixed blend is the

## Exercise 8 - Challenge: your own weighting of the 8-path blend, placed on the frontier

The dense-heavy weighting lands at about 0.376 / 0.1835 and joins the frontier (no other point has both more hit and more reach). My own goal is **reach at the full stack's hit**: I zero popularity (the most head-heavy path), halve MF, and lean on item-item and semantic, the two widest paths. It gives about 0.356 / 0.267, on the frontier: the full stack's hit with about 0.065 more coverage.

In [9]:
DENSE_HEAVY = {**{name: 1.0 for name in FULL8_PATH_NAMES},
               "popularity": 0.25, "lexical": 0.5, "semantic": 0.5, "two-tower": 1.5, "hybrid": 1.5}
dense_heavy = system.evaluate(FixedBlendPath(system.full8, DENSE_HEAVY))
points_with_dense = {**points, "dense-heavy": (dense_heavy["hit"], dense_heavy["coverage"])}
dense_on_frontier = bool(not dominated("dense-heavy", points_with_dense))

# My goal: MORE REACH at the full stack's hit -- drop the head-heavy popularity path, halve MF, and
# give more weight to the two widest-reaching paths (item-item CF and semantic).
MY_WEIGHTS = {**{name: 1.0 for name in FULL8_PATH_NAMES},
              "popularity": 0.0, "mf": 0.5, "item-item": 1.5, "semantic": 1.5}
my_readout = system.evaluate(FixedBlendPath(system.full8, MY_WEIGHTS))
points_with_mine = {**points_with_dense, "mine": (my_readout["hit"], my_readout["coverage"])}
my_on_frontier = bool(not dominated("mine", points_with_mine))

weights_explanation = (
    "My goal was more reach at the full stack's hit: with popularity off, MF halved and item-item and "
    "semantic at 1.5, the blend scores about 0.356 hit with coverage about 0.267 -- the full stack's "
    "hit with about 0.065 more reach, and on the frontier. Dense-heavy goes the other way, buying hit "
    "(0.376) with reach (0.184). A few hand-set weights move the system ALONG the frontier further, "
    "in either direction, than the learned reranker moved it at all, so on this data the reranker's "
    "extra machinery has not justified itself against a simple, explainable blend."
)

print(f"dense-heavy: hit {dense_heavy['hit']:.3f}, coverage {dense_heavy['coverage']:.4f}, "
      f"head-share {dense_heavy['head_share']:.3f}, Gini {dense_heavy['exposure_gini']:.3f} | "
      f"on the frontier: {dense_on_frontier}")
print(f"mine: hit {my_readout['hit']:.3f}, coverage {my_readout['coverage']:.4f}, "
      f"head-share {my_readout['head_share']:.3f}, Gini {my_readout['exposure_gini']:.3f} | "
      f"on the frontier: {my_on_frontier}")
print(f"frontier with both: {[n for n in points_with_mine if not dominated(n, points_with_mine)]}")
print(weights_explanation)

assert dense_on_frontier and abs(dense_heavy["hit"] - 0.376) < 0.0011 and abs(dense_heavy["coverage"] - 0.1835) < 0.0011
assert my_on_frontier, "the reach-leaning weighting lands on the frontier"
assert my_readout["hit"] >= points["full-stack"][0] and my_readout["coverage"] > points["full-stack"][1] + 0.05, \
    "it matches the full stack's hit with clearly more reach"
assert dominated("full-stack", points_with_mine), "the full stack stays inside the frontier"

dense-heavy: hit 0.376, coverage 0.1835, head-share 0.885, Gini 0.926 | on the frontier: True
mine: hit 0.356, coverage 0.2670, head-share 0.763, Gini 0.901 | on the frontier: True
frontier with both: ['part1-build', 'fixed-blend', 'content-reranker', 'dense-heavy', 'mine']
My goal was more reach at the full stack's hit: with popularity off, MF halved and item-item and semantic at 1.5, the blend scores about 0.356 hit with coverage about 0.267 -- the full stack's hit with about 0.065 more reach, and on the frontier. Dense-heavy goes the other way, buying hit (0.376) with reach (0.184). A few hand-set weights move the system ALONG the frontier further, in either direction, than the learned reranker moved it at all, so on this data the reranker's extra machinery has not justified itself against a simple, explainable blend.


## Exercise 9 - Challenge: one ranking feature closes the door on cold books

In the cached training matrix the held-out positives have a mean `log_pop` of about 0.548 against 0.420 for the negatives, so the reranker learns that popular books rank higher. Dropping that single column and refitting trades about 0.018 hit for about 0.036 coverage and a lower head-share, and it reopens the door: 29 zero-train books reach the cohort's top-10s, against none for the full stack.

In [10]:
log_pop_column = names.index("log_pop")
log_pop_means = (float(X[y == 1, log_pop_column].mean()), float(X[y == 0, log_pop_column].mean()))

without_log_pop_mask = np.array([name != "log_pop" for name in names])
without_log_pop = with_model(RerankerModel(seed=SEED).fit(X, y, feature_mask=without_log_pop_mask))
without_log_pop_lists = system.top_lists(without_log_pop)
without_log_pop_readout = system.readout(without_log_pop_lists)
without_log_pop_cold = len(cold_in(without_log_pop_lists))

full_stack_hit = baselines["full-stack"]["hit"]
reopens_cold = bool(without_log_pop_cold > 0 and cold_slice["full stack"] == 0)
costs_hit = bool(without_log_pop_readout["hit"] < full_stack_hit)

log_pop_explanation = (
    "One feature controls the trade: without log_pop the ranker gives up about 0.018 hit (0.356 -> "
    "0.338) for about 0.036 more coverage, a head-share of 0.79 instead of 0.85, and 29 cold books "
    "where the full stack showed none. The held-out reads are drawn from a log in which popular books "
    "were more visible, so positives lean popular (0.548 vs 0.420) and any ranker trained on that log "
    "keeps learning popularity, whether from this column or from a proxy for it. I would ship the "
    "log_pop-free ranker only for a reach goal -- surfacing new books and new authors -- and keep the "
    "full one when hit@10 is the target."
)

print(f"log_pop is column {log_pop_column} | mean positive {log_pop_means[0]:.3f} vs negative {log_pop_means[1]:.3f}")
print(f"without log_pop: hit {without_log_pop_readout['hit']:.3f}, coverage {without_log_pop_readout['coverage']:.4f}, "
      f"head-share {without_log_pop_readout['head_share']:.3f} | cold books {without_log_pop_cold} "
      f"(full stack {cold_slice['full stack']})")
print(f"reopens cold: {reopens_cold} | costs hit: {costs_hit}")
print(log_pop_explanation)

assert log_pop_column == 20 and log_pop_means[0] > log_pop_means[1] + 0.1
assert reopens_cold and without_log_pop_cold == 29
assert costs_hit and without_log_pop_readout["coverage"] > points["full-stack"][1]
assert without_log_pop_readout["head_share"] < 0.8 and abs(without_log_pop_readout["hit"] - 0.338) < 0.0011

log_pop is column 20 | mean positive 0.548 vs negative 0.420
without log_pop: hit 0.338, coverage 0.2385, head-share 0.793 | cold books 29 (full stack 0)
reopens cold: True | costs hit: True
One feature controls the trade: without log_pop the ranker gives up about 0.018 hit (0.356 -> 0.338) for about 0.036 more coverage, a head-share of 0.79 instead of 0.85, and 29 cold books where the full stack showed none. The held-out reads are drawn from a log in which popular books were more visible, so positives lean popular (0.548 vs 0.420) and any ranker trained on that log keeps learning popularity, whether from this column or from a proxy for it. I would ship the log_pop-free ranker only for a reach goal -- surfacing new books and new authors -- and keep the full one when hit@10 is the target.
